<h1>Q-Learning</h1>

<p>In this exercise, you will study the stochastic Q-Learning algorithm using the ready-made program provided.</p>

<p>This example presents a Q-Learning implementation for an autonomous taxi-driving scenario. In this problem, the taxi must satisfy the following requirements:</p>

<ul>
<li>Drop off the passenger at the correct destination.</li>
<li>Follow the shortest possible route.</li>
<li>Respect traffic rules and ensure passenger safety.</li>
</ul>

<p>The problem is defined in terms of rewards, states, and actions, as described below.</p>

<h4>Reward</h4>
<ul>
<li>The taxi receives the highest reward when it successfully drops off the passenger at the correct destination.</li>
<li>A penalty is applied if the taxi drops off the passenger at the wrong location.</li>
<li>A small penalty is applied at each step to encourage the taxi to reach the destination as quickly as possible.</li>
</ul>

<p>These reward rules can be summarized as follows: the taxi receives <b>20 points</b> when it successfully drops off the passenger at the correct destination and loses <b>1 point</b> for each time step taken. In addition, a <b>10-point penalty</b> is applied when the taxi attempts to pick up or drop off the passenger at an incorrect location.</p>

<h4>Number of States</h4>
<img src="https://storage.googleapis.com/lds-media/images/Reinforcement_Learning_Taxi_Env.width-1200.png">

<p>In this example, the taxi operates in a small <b>5 × 5 grid</b>. The environment also includes <b>4 possible destination locations</b> and <b>5 possible passenger locations</b>.</p>

<p>Therefore, the total number of possible states is calculated as:</p>

<p><b>5 × 5 × 5 × 4 = 500 states</b></p>

<p>Each state represents a different combination of the taxi's position, the passenger's location, and the passenger's destination.</p>

<h4>Number of Actions</h4>

<p>The taxi can choose from <b>six possible actions</b>:</p>

<ul>
<li><b>0</b> = Move South</li>
<li><b>1</b> = Move North</li>
<li><b>2</b> = Move East</li>
<li><b>3</b> = Move West</li>
<li><b>4</b> = Pick up the passenger</li>
<li><b>5</b> = Drop off the passenger</li>
</ul>

<p>However, not every action is valid in every state. For example, if the taxi is next to a wall or obstacle in the grid, it may not be able to move in that direction. Similarly, pickup and drop-off actions are only appropriate when the taxi is at the correct location.</p>





<p>Before continuing with the exercise, answer the following questions:</p>
<ul>
<li>Is Q-Learning a model-free or model-based reinforcement learning algorithm? Justify your answer by explaining whether the agent needs prior knowledge of the environment’s transition probabilities and reward function.</li>
<li>Briefly describe the Q-Learning algorithm. In which problems do you think Reinforcement Learning is suitable as a learning approach? What is the main difference between the Q-Learning algorithm and the Policy Iteration and Value Iteration algorithms?</li>
</ul>

<p>Next, you need to load the <i>gymnasium</i> library as well as the relevant dataset.</p>

In [ ]:
!pip install gymnasium

In [ ]:
import gymnasium as gym

env = gym.make("Taxi-v3", render_mode="rgb_array")
# Reset the environment to start
env.reset()

# Render the initial state
env.render()

In [ ]:
env.reset() # reset environment to a new, random state

print("Action Space {}".format(env.action_space))
print("State Space {}".format(env.observation_space))

# Render the initial state
env.render()

<p>Below, we define the taxi’s coordinates, the passenger’s location, and the destination point.</p>

In [ ]:
taxi = env.unwrapped            # the actual TaxiEnv


state = taxi.encode(3, 1, 2, 0) # (taxi row, taxi column, passenger index, destination index)
print("State:", state)

taxi.s = state

<p>Below is the reward matrix for the state we defined in the previous step.</p>

In [ ]:
taxi.P[328]

<p>We run our example without using Q-Learning.</p>

<ul><li>Are the results satisfactory? How would the use of Q-Learning help us?</li></ul>

In [ ]:
taxi.s = 328  # set environment to illustration's state

epochs = 0
penalties, reward = 0, 0

frames = [] # for animation

done = False

while not done:
    action = taxi.action_space.sample()
    state, reward, done, truncated, info = env.step(action)

    if reward == -10:
        penalties += 1

    # Put each rendered frame into dict for animation
    frames.append({
        'frame': taxi.render(),
        'state': state,
        'action': action,
        'reward': reward
        }
    )

    epochs += 1


print("Timesteps taken: {}".format(epochs))
print("Penalties incurred: {}".format(penalties))

<p>Now we will try to solve our problem using Q-Learning.</p>

<ul><li>What is the role of the parameters α and γ in Q-Learning? What would be the effect of setting both parameters equal to 1?</li></ul>

In [ ]:
import numpy as np
q_table = np.zeros([taxi.observation_space.n, taxi.action_space.n])

In [ ]:
%%time
"""Training the agent"""

import random
from IPython.display import clear_output

# Hyperparameters
alpha = 0.1
gamma = 0.6
epsilon = 0.1

# For plotting metrics
all_epochs = []
all_penalties = []

for i in range(1, 100001):
    state, _ = taxi.reset()

    epochs, penalties, reward, = 0, 0, 0
    done = False

    while not done:
        if random.uniform(0, 1) < epsilon:
            action = env.action_space.sample() # Explore action space
        else:
            action = np.argmax(q_table[state]) # Exploit learned values

        next_state, reward, done, truncated, info = taxi.step(action)

        old_value = q_table[state, action]
        next_max = np.max(q_table[next_state])

        new_value = (1 - alpha) * old_value + alpha * (reward + gamma * next_max)
        q_table[state, action] = new_value

        if reward == -10:
            penalties += 1

        state = next_state
        epochs += 1

    if i % 100 == 0:
        clear_output(wait=True)
        print(f"Episode: {i}")

print("Training finished.\n")

In [ ]:
q_table[328]

In [ ]:
"""Evaluate agent's performance after Q-learning"""

total_epochs, total_penalties = 0, 0
episodes = 100

for _ in range(episodes):
    state, _ = taxi.reset()
    epochs, penalties, reward = 0, 0, 0

    done = False

    while not done:
        action = np.argmax(q_table[state])
        state, reward, done, truncated, info = env.step(action)

        if reward == -10:
            penalties += 1

        epochs += 1

    total_penalties += penalties
    total_epochs += epochs

print(f"Results after {episodes} episodes:")
print(f"Average timesteps per episode: {total_epochs / episodes}")
print(f"Average penalties per episode: {total_penalties / episodes}")

<p>Evaluate and compare the two algorithms using the following performance metrics:</p>

<ul>
<li>Average number of penalties per episode</li>
<li>Average number of steps required to complete a trip</li>
<li>Average reward obtained per move</li>
</ul>

<p>Run the comparison over <b>100 episodes</b>.</p>